# Exercise: Feature Engineering on a Real Dataset

## Predicting Diamond Prices

The snowman data had 14 rows. In this exercise you work with a real dataset of almost 54,000 diamonds. Your goal is to predict the **price** of a diamond from its weight, its dimensions and three quality grades.

You follow the [machine learning workflow](../machine_learning_workflow.md) from start to end, in the same steps as the diabetes project: split the data into training, validation and test sets, compare several versions of a model on the validation set, and score only the chosen version on the test set. Each version adds one feature engineering idea, so the validation set tells you which ideas help.

<img src="../assets/ml_workflow.png" width="600">

## Learning Objectives

At the end of this exercise, you should be able to:

- Split a dataset into training, validation and test sets, and explore only the training set.
- Encode ordered quality grades with `OrdinalEncoder` and with `OneHotEncoder`, and compare the two on a validation set.
- Log-transform skewed features and a skewed target, and convert the predictions back to the original units.
- Retrain the chosen version on the training and validation sets and score it once on the test set.

## Setup

Documentation: [`set_config`](https://scikit-learn.org/stable/modules/generated/sklearn.set_config.html).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
    StandardScaler,
)

sns.set_style("whitegrid")

# Transformers return pandas DataFrames instead of NumPy arrays
set_config(transform_output="pandas")

## Step 1: Define the Goal and the Metric

**Goal:** predict the price of a diamond in US dollars.

**Metric:** the root mean squared error ([`root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)), in dollars. It reads as "on average, the prediction is about this many dollars off". We also report $R^2$. A model is only useful if it clearly beats a **baseline** that predicts the average price of the training diamonds for every diamond.

## Step 2: Get the Data

The data is the `diamonds` dataset that ships with the R package [ggplot2](https://ggplot2.tidyverse.org/reference/diamonds.html).

| Column | Description |
|---|---|
| `price` | price in US dollars (the target) |
| `carat` | weight of the diamond (1 carat = 0.2 g) |
| `cut` | quality of the cut, from worst to best: `Fair`, `Good`, `Very Good`, `Premium`, `Ideal` |
| `color` | colour grade, from worst to best: `J`, `I`, `H`, `G`, `F`, `E`, `D` |
| `clarity` | clarity grade, from worst to best: `I1`, `SI2`, `SI1`, `VS2`, `VS1`, `VVS2`, `VVS1`, `IF` |
| `depth` | total depth as a percentage of the average width |
| `table` | width of the flat top as a percentage of the widest point |
| `x`, `y`, `z` | length, width and depth in mm |

`cut`, `color` and `clarity` are **ordinal**: their categories have a natural order.

In [ ]:
df = pd.read_csv("../data/diamonds.csv")
df.head()

In [ ]:
df.info()

## Step 3: Split into Training, Validation and Test Sets

Split **before** you look closely at the data, exactly as in the diabetes project: first set aside 33% of the diamonds as the test set, then 25% of the rest as the validation set. Use `random_state=42` so your numbers match the solution. There is no `stratify` here, because the target is a number, not a class.

Documentation: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
df_train_val, df_test = train_test_split(df, test_size=0.33, random_state=42)

df_train, df_val = train_test_split(df_train_val, test_size=0.25, random_state=42)

print(f"Training set:   {df_train.shape[0]} rows")
print(f"Validation set: {df_val.shape[0]} rows")
print(f"Test set:       {df_test.shape[0]} rows")

## Step 4: Explore the Training Set

Explore only `df_train`. What you find here guides your feature engineering.

`df.info()` showed no missing values. Look at the summary statistics of the numeric columns: does every value make sense for a real diamond?

In [ ]:
df_train.describe().round(2)

Two things stand out:

- The minimum of `x`, `y` and `z` is 0.00: a diamond with no length, width or depth cannot exist. These are recording errors.
- `price` has a mean (3,958 dollars) far above its median (2,400 dollars), and a maximum of 18,818 dollars: a few very expensive diamonds pull the mean up. The maximum `y` of 58.90 mm is also suspicious, as 75% of the training diamonds are at most 6.54 mm wide.

Count the training diamonds with a zero in `x`, `y` or `z`.

In [ ]:
(df_train[["x", "y", "z"]] == 0).any(axis=1).sum()

13 of the 27,104 training diamonds have at least one zero dimension. They are few, but a zero in a feature the model relies on gives a badly wrong prediction for that diamond.

Plot the distribution of `price`. Is it symmetric?

In [ ]:
sns.histplot(data=df_train, x="price");

The distribution is strongly **right-skewed**: most diamonds cost less than about 5,000 dollars, with a long tail of expensive ones. A linear model fitted to a skewed target tends to fit the many cheap diamonds poorly, and nothing stops it from predicting prices below zero.

Plot `price` against `carat`. The training set has about 27,000 points, so make them small and transparent (`s=5`, `alpha=0.1`). Is the relationship a straight line?

In [ ]:
sns.scatterplot(data=df_train, x="carat", y="price", s=5, alpha=0.1);

The relationship is not a straight line: the price rises steeply with `carat`, and the spread of prices grows with it. The weights also pile up at round values: 2,782 training diamonds weigh between 1.00 and 1.05 carats, but only 153 between 0.95 and 1.00. Sellers prefer a diamond that just reaches a round weight.

Calculate the median price for each `cut`. Which cut is the cheapest, and why could that be? Hint: compare the median `carat` of each cut as well.

In [ ]:
df_train.groupby("cut")[["price", "carat"]].median()

`Ideal`, the best cut, has the lowest median price (1,804.5 dollars). The median `carat` explains it: `Ideal` diamonds are the smallest (0.54 carats against 1.00 for `Fair`). Weight drives the price so strongly that it hides the effect of the cut. A model that sees `carat` and `cut` together can separate the two effects.

## Step 5: Mark the Impossible Values as Missing

A zero length, width or depth is a recording error, not a real measurement. Replace the zeros in `x`, `y` and `z` with missing values (`np.nan`) in **all three** sets, so that an imputer can fill them later.

This replacement does not leak information: it applies a fixed rule to each row and learns nothing from the data. The imputer **learns** a median, so it belongs in the pipeline and is fitted on the training set only.

In [ ]:
def zeros_to_missing(data):
    """Return a copy of data with the zeros in x, y and z replaced by np.nan."""
    data = data.copy()
    data[["x", "y", "z"]] = data[["x", "y", "z"]].replace(0, np.nan)
    return data


df_train = zeros_to_missing(df_train)
df_val = zeros_to_missing(df_val)
df_test = zeros_to_missing(df_test)
df_train[["x", "y", "z"]].isna().sum()

Separate the features from the target in each set.

In [ ]:
target = "price"

X_train, y_train = df_train.drop(columns=target), df_train[target]
X_val, y_val = df_val.drop(columns=target), df_val[target]
X_test, y_test = df_test.drop(columns=target), df_test[target]

## Steps 6 to 8: Train and Compare Model Versions

Every version is a [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) with a [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) followed by a [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html). Each version changes one feature engineering idea. Before you build the versions, complete the helper `evaluate`, which you will call once for every version. The `else` branch, for a price in dollars, is already written: use it as the model for the `log_target` branch. The helper should:

1. Fit the model on the training set.
2. Predict the training **and** the validation set, so you can compare the two scores and spot overfitting.
3. Calculate the RMSE on both sets and $R^2$ on the validation set.
4. Report the **lowest** predicted price: a price below zero is a sign that the model does not fit the data well.

When `log_target=True`, the model must learn the **logarithm** of the price: fit it on `np.log(y_train)`, then convert its predictions back to dollars with `np.exp`, so that every score is in dollars. You will need this in version 4.

To check your helper: version 1 should reach a validation RMSE of about 1,487 dollars.

Documentation: [`r2_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).

In [ ]:
scores = []


def evaluate(name, model, log_target=False):
    """Fit model on the training set, score it in dollars on the training and validation sets."""
    if log_target:
        model.fit(X_train, np.log(y_train))
        train_pred = np.exp(model.predict(X_train))
        val_pred = np.exp(model.predict(X_val))
    else:
        model.fit(X_train, y_train)
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
    scores.append(
        {
            "version": name,
            "train RMSE": root_mean_squared_error(y_train, train_pred),
            "validation RMSE": root_mean_squared_error(y_val, val_pred),
            "validation R2": r2_score(y_val, val_pred),
            "lowest prediction": val_pred.min(),
        }
    )
    return pd.DataFrame(scores).round(3)

### Baseline

The baseline predicts the training mean for every diamond. Its RMSE is the number every model must beat.

In [ ]:
baseline_pred = pd.Series(y_train.mean(), index=y_val.index)
root_mean_squared_error(y_val, baseline_pred)

Predicting the training mean misses the validation diamonds by about 3,979 dollars on average. Every version below must clearly beat this number.

### Version 1: Numeric Features

Use the six numeric columns. `x`, `y` and `z` now contain missing values, so every numeric column goes through a pipeline that imputes the median and then standardises.

Documentation: [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html), [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [ ]:
numeric = ["carat", "depth", "table", "x", "y", "z"]

numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

v1 = Pipeline(
    steps=[
        ("features", ColumnTransformer(transformers=[("numeric", numeric_pipeline, numeric)])),
        ("model", LinearRegression()),
    ]
)
evaluate("1 numeric", v1)

The numeric features cut the validation RMSE from 3,979 to 1,487 dollars ($R^2$ 0.860). The training and validation RMSE are almost equal: with 27,104 training diamonds and only 6 features, the model is too simple to overfit. Its lowest prediction, however, is -2,858 dollars.

### Version 2: Add the Grades with `OrdinalEncoder`

The three grades are ordinal, so encode them with [`OrdinalEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html). Pass the order of each grade, from worst to best, with the `categories` argument: otherwise the encoder sorts the categories alphabetically, and `Fair < Good < Ideal < Premium < Very Good` is not the order of the cut.

In [ ]:
grades = ["cut", "color", "clarity"]
grade_order = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],
]

ordinal_encoder = OrdinalEncoder(categories=grade_order)

v2 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("numeric", numeric_pipeline, numeric),
                    ("grades", ordinal_encoder, grades),
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("2 + ordinal grades", v2)

The grades lower the validation RMSE to 1,226 dollars ($R^2$ 0.905): two diamonds of the same weight can differ a lot in price because of their cut, colour and clarity.

### Version 3: One-Hot Encode the Grades Instead

`OrdinalEncoder` turns each grade into 0, 1, 2, ..., so the linear model gives every step up the scale the **same** price change. One-hot encoding lets each grade have its own price change instead. Replace the ordinal encoder with a [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [ ]:
onehot_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)

v3 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("numeric", numeric_pipeline, numeric),
                    ("grades", onehot_encoder, grades),
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("3 one-hot grades", v3)

One-hot encoding beats ordinal encoding: 1,151 against 1,226 dollars. The steps between grades are not equal in price, and one-hot encoding lets the model learn a separate price change for each grade instead of a single change per step.

The lowest prediction has become worse, though: -22,645 dollars. It belongs to a validation diamond recorded with `z` = 31.8 mm, almost certainly a typing error for 3.18 mm. A linear model extends its straight lines to such extreme values without limit. Version 3 predicts a negative price for 640 of the 9,035 validation diamonds, mostly small ones (their median weight is 0.32 carats). A straight line through the price cannot bend to stay above zero for the smallest diamonds.

### Version 4: Log-Transform the Skewed Columns

Your plots showed a skewed price and a curved relationship with `carat`. Taking the logarithm of both straightens the curve and stops the model from predicting negative prices, because the exponential of any number is positive.

- Give `carat`, `x`, `y` and `z` a pipeline that imputes, applies `np.log` with a [`FunctionTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html), then scales.
- Keep `depth` and `table` in the numeric pipeline, and keep the one-hot encoded grades.
- Call `evaluate` with `log_target=True`, so the model learns the logarithm of the price.

In [ ]:
skewed = ["carat", "x", "y", "z"]
other = ["depth", "table"]

log_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(func=np.log, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
    ]
)

v4 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("skewed", log_pipeline, skewed),
                    ("other", numeric_pipeline, other),
                    ("grades", onehot_encoder, grades),
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("4 + log transform", v4, log_target=True)

The log transform gives by far the largest gain: the validation RMSE drops from 1,151 to 786 dollars and $R^2$ rises to 0.961. On the log scale the relationship between weight and price is close to a straight line, which is what a linear model can fit. The lowest prediction is now 272 dollars: converting with `np.exp` can only give positive prices.

### Choose a Version

Which version has the lowest validation RMSE? Is its training RMSE close to its validation RMSE, or does it overfit? Does it ever predict a negative price? Write down your choice before you continue.

**Answer:** version 4 has the lowest validation RMSE (786 dollars, $R^2$ 0.961) and never predicts a negative price. Its training RMSE (817 dollars) is close to its validation RMSE, so it does not overfit.

## Step 9: Calculate the Test Score

As in the diabetes project, retrain the chosen version on the training and validation sets **together**, with the same settings, then score it **once** on the test set. Remember to train on the logarithm of the price and to convert the predictions back with `np.exp`. Score the baseline on the test set too, for comparison.

In [ ]:
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

v4.fit(X_train_val, np.log(y_train_val))

test_pred = np.exp(v4.predict(X_test))
print(f"Test RMSE: {root_mean_squared_error(y_test, test_pred):.0f} dollars  Test R2: {r2_score(y_test, test_pred):.3f}")

baseline_test = pd.Series(y_train_val.mean(), index=y_test.index)
print(f"Baseline test RMSE: {root_mean_squared_error(y_test, baseline_test):.0f} dollars")

The chosen version reaches a test RMSE of 800 dollars and an $R^2$ of 0.959, against 3,952 dollars for the baseline. The test score is very close to the validation score (786 dollars): with about 9,000 validation diamonds, the validation set gave an honest estimate of how the model performs on new data.

## Bonus: Polynomial Terms Instead of the Logarithm

[`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) is another way to follow a curve. Build a version 5 from version 3 (no log transform, the price in dollars), but give `carat` its own pipeline that imputes, scales and adds degree-3 polynomial terms. Compare it with version 4 on the validation set, including the lowest prediction.

Use the training and validation sets only: the test set has already been used for the final score.

In [ ]:
polynomial_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("polynomial", PolynomialFeatures(degree=3, include_bias=False)),
    ]
)

v5 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("carat", polynomial_pipeline, ["carat"]),
                    ("numeric", numeric_pipeline, ["depth", "table", "x", "y", "z"]),
                    ("grades", onehot_encoder, grades),
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("5 polynomial carat", v5)

The polynomial terms improve on version 3 (1,112 against 1,151 dollars) but stay far behind the log transform (786 dollars). They do not fix the negative predictions either: 592 validation diamonds still get a price below zero, and the lowest prediction (-24,417 dollars) again belongs to the diamond recorded with `z` = 31.8 mm, because `z` still enters the model on its original scale. The logarithm suits this data better: the price grows by a similar **percentage** for each step up in weight, and on the log scale that becomes a straight line. Version 4 remains the better choice.

## Bonus 2: Feature Selection

Version 4 uses 23 columns. In W3D2 you learned to ask whether all of them are needed. The question matters here because `carat`, `x`, `y` and `z` all measure the **size** of a diamond.

Start with a filter method: fit the `ColumnTransformer` of version 4 on the training set and look at the correlations between its six numeric output columns.

In [ ]:
features_train = v4.named_steps["features"].fit_transform(X_train)

sns.heatmap(features_train.iloc[:, :6].corr(), annot=True, fmt=".3f", cmap="coolwarm", center=0);

After the log transform, `carat`, `x`, `y` and `z` are almost the same variable: their correlations range from 0.990 to 0.998. The weight of a diamond follows from its dimensions, so the three dimensions add almost nothing once `carat` is in the model. `depth` and `table` are only weakly correlated with the size columns (at most 0.197 in absolute value).

Compare two smaller versions with version 4 on the validation set. Both keep the log-transformed `carat`, the one-hot encoded grades and the log target:

- **Version 6** drops `x`, `y` and `z`.
- **Version 7** also drops `depth` and `table`, so it keeps only `carat` and the grades.

Then test whether the grades are needed at all with **version 8**: version 4 without the grades.

In [ ]:
v6 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("skewed", log_pipeline, ["carat"]),
                    ("other", numeric_pipeline, other),
                    ("grades", onehot_encoder, grades),
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("6 without x, y, z", v6, log_target=True)

v7 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("skewed", log_pipeline, ["carat"]),
                    ("grades", onehot_encoder, grades),
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("7 carat and grades only", v7, log_target=True)

v8 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("skewed", log_pipeline, skewed),
                    ("other", numeric_pipeline, other),
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("8 without grades", v8, log_target=True)

- **Version 6** drops `x`, `y` and `z` and is slightly **better** than version 4: 778 against 786 dollars. The near-duplicate columns did not help the model, and they make its coefficients hard to interpret, because four columns share the effect of size.
- **Version 7** also drops `depth` and `table` and scores the same (778 dollars) with 18 columns instead of 23. A smaller model that is just as accurate is easier to explain and maintain. It also no longer needs the zero cleaning of Step 5, and the diamond recorded with `z` = 31.8 mm cannot mislead it.
- **Version 8** drops the grades and doubles the error to 1,608 dollars: the grades carry information that the size cannot replace.

The test set has already been used for the final score, so these versions are compared on the validation set only. In a real project, feature selection belongs to Steps 6 to 8, **before** you choose a version and score it on the test set.

Finally, try a wrapper method. [`RFE`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html) can sit inside the pipeline, between the `ColumnTransformer` and the model, and keep the 10 output columns it ranks highest. Compare its validation RMSE with version 7, which uses 18 columns (`carat` and the 17 one-hot grade columns), and look at which columns RFE kept.

In [ ]:
v9 = Pipeline(
    steps=[
        ("features", v4.named_steps["features"]),
        ("selector", RFE(LinearRegression(), n_features_to_select=10)),
        ("model", LinearRegression()),
    ]
)
evaluate("9 RFE, 10 columns", v9, log_target=True)

In [ ]:
# The columns RFE kept
v9.named_steps["selector"].get_feature_names_out()

RFE does clearly worse: 1,059 dollars against 778 for version 7. It ranks the **individual** output columns, so it keeps some one-hot columns of a grade and drops others: it keeps all seven clarity columns but only one of the six colour columns (`color_J`), and no cut column. Dropping the five columns `color_E` to `color_I` merges those grades with the reference grade `D`: the model gives an `E`, `F`, `G`, `H` or `I` diamond the same price as a `D` diamond of the same size and clarity, although colour is one of the grades that set the price.

When a feature is one-hot encoded, select it as a **whole group**, as you did with versions 6 to 8.

### Create a Feature by Hand: Volume

Instead of dropping `x`, `y` and `z`, you can **combine** them into one feature that has a physical meaning: the volume of the diamond in cubic millimetres, `x * y * z`. Add it to the training and the validation set (the test set has already been used). As in Step 5, this is a fixed rule for each row, so it learns nothing and leaks nothing.

How strongly is the logarithm of the volume correlated with the logarithm of `carat`?

In [ ]:
for X in (X_train, X_val):
    X["volume"] = X["x"] * X["y"] * X["z"]

np.log(X_train["volume"]).corr(np.log(X_train["carat"]))

The correlation is 0.999: on the log scale, volume and weight are almost the same variable. That makes sense physically, because the weight of a diamond is its volume times the density of diamond, which is the same for every stone.

Weight and volume measure the same thing, so the volume could replace `carat`. Build **version 10**: version 7 with `volume` in place of `carat`, and compare. Look at the training RMSE as well as the validation RMSE.

In [ ]:
v10 = Pipeline(
    steps=[
        (
            "features",
            ColumnTransformer(
                transformers=[
                    ("skewed", log_pipeline, ["volume"]),
                    ("grades", onehot_encoder, grades),
                ]
            ),
        ),
        ("model", LinearRegression()),
    ]
)
evaluate("10 volume instead of carat", v10, log_target=True)

Version 10 is much worse than version 7: a validation RMSE of 1,643 against 778 dollars. Even stranger, its **training** RMSE (8,602 dollars) is far above its validation RMSE. A model normally fits the data it was trained on better than new data, so a few training diamonds must have enormous errors.

A training RMSE far **above** the validation RMSE is unusual. Find the training diamond with the largest error and look at its dimensions.

In [ ]:
train_pred = pd.Series(np.exp(v10.predict(X_train)), index=X_train.index)

errors = (train_pred - y_train).abs()

X_train.loc[[errors.idxmax()], ["carat", "x", "y", "z", "volume"]].assign(price=y_train, predicted=train_pred)

The largest error belongs to the diamond recorded with `y` = 58.9 mm, the suspicious maximum from Step 4. Its other two dimensions are about 8 mm, so a width of 58.9 mm cannot be right. The error makes its volume 3,841 cubic millimetres, more than 30 times the median, and the model predicts a price of about 1,421,330 dollars for a diamond that costs 12,210. A single prediction this far off is enough to dominate the training RMSE. The validation set has a similar case: the diamond with `z` = 31.8 mm is predicted at 137,607 dollars instead of 1,970.

Version 7 predicts 13,057 and 1,735 dollars for the same two diamonds, because `carat` is measured directly on a scale and does not contain the typing errors. The lesson for features created by hand: a new feature **inherits the errors of its parts**, and multiplying three numbers turns one wrong digit into a huge value. Before you trust a created feature, check its extremes, as you checked the summary statistics in Step 4.

## Summary

In this exercise you:

- Split a real dataset into training, validation and test sets, and explored only the training set.
- Marked impossible values as missing and imputed them inside a pipeline fitted on the training set.
- Compared ordinal and one-hot encoding of quality grades on the validation set.
- Log-transformed skewed features and the target, which improved the model and removed negative predictions.
- Retrained the chosen version on the training and validation sets and scored it once on the test set.
- Selected features as whole groups, created a volume feature by hand, and compared these versions on the validation set (bonus).

With about 9,000 validation diamonds, the comparison of versions is already stable. On smaller datasets it is not, which is why cross-validation in W3D4 replaces the single validation set.